# 🤾 Balonmano IA en Google Colab

Pone en marcha la app completa (dashboard + análisis con IA) en una máquina
gratuita de Google **con GPU**, y te da un **enlace público** para abrirla
desde el móvil o el ordenador.

**Cómo usarlo**

1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** → Guardar.
2. Rellena la configuración del paso 1 (o déjala como está).
3. Menú **Entorno de ejecución → Ejecutar todas**. Tarda ~3-5 minutos la primera vez.
4. Al final del paso 3 aparece el **enlace** y la **contraseña**. ¡Listo!

**A tener en cuenta**

- La app funciona mientras esta pestaña siga abierta (Colab gratuito corta tras
  unas horas o si no la usas un rato). Para volver a usarla, repite *Ejecutar todas*:
  el enlace cambia cada vez.
- Con *Guardar en Drive* activado, las sesiones analizadas se guardan en tu Google
  Drive (carpeta `BalonmanoIA`) y siguen ahí la próxima vez.
- Por el enlace público se pueden subir vídeos de **hasta ~100 MB**. Para vídeos más
  grandes (partidos completos), usa el paso 4: los copias a Drive y se analizan desde aquí.
- **En directo con el móvil**: en la app, *Nueva sesión → En directo → Cámara de este
  dispositivo*. El móvil envía la imagen a Colab y ves las estadísticas al momento.
  Las cámaras IP (RTSP) de la red del pabellón no funcionan desde Colab.

In [ ]:
#@title 1. Configuración { display-mode: "form" }
#@markdown Contraseña para entrar en la app (déjala vacía para generar una aleatoria).
CONTRASENA = ""  #@param {type:"string"}
#@markdown Guardar vídeos y resultados en tu Google Drive (recomendado).
GUARDAR_EN_DRIVE = True  #@param {type:"boolean"}
#@markdown Rama del repositorio a usar.
RAMA = "claude/handball-training-analysis-app-35er51"  #@param {type:"string"}

import secrets

REPO_URL = "https://github.com/CarlosJavierML/Balonmano_ia.git"
if not CONTRASENA:
    CONTRASENA = secrets.token_urlsafe(9)
print("Configuración lista.")

In [ ]:
#@title 2. Instalar (~3-5 minutos) { display-mode: "form" }
import os, shutil, subprocess, sys, urllib.request
from pathlib import Path

def run(cmd, cwd=None, env=None):
    """Runs a shell command, showing its output only if it fails."""
    result = subprocess.run(cmd, shell=True, cwd=cwd, env=env, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError(f"Falló: {cmd}")
    return result.stdout

BASE = Path(os.environ.get("BALONMANO_COLAB_BASE", "/content"))
APP_DIR = BASE / "Balonmano_ia"

# Datos: en Drive (persisten entre sesiones) o en el disco temporal de Colab.
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BalonmanoIA")
else:
    DATA_DIR = BASE / "balonmano_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / "importar").mkdir(exist_ok=True)

print("1/5 Descargando la app…")
if (APP_DIR / ".git").exists():
    run(f"git fetch -q --depth 1 origin {RAMA} && git checkout -q -B {RAMA} FETCH_HEAD", cwd=APP_DIR)
else:
    run(f"git clone -q --depth 1 -b {RAMA} {REPO_URL} {APP_DIR}")

print("2/5 Instalando dependencias de Python…")
# Colab ya trae PyTorch con soporte de GPU: no se reinstala.
run(f"{sys.executable} -m pip install -q -r backend/requirements.txt", cwd=APP_DIR)

print("3/5 Construyendo el dashboard…")
def node_major():
    try:
        return int(run("node --version").strip().lstrip("v").split(".")[0])
    except Exception:
        return 0
if node_major() < 18:
    run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y -qq nodejs")
run("npm ci --no-audit --no-fund --loglevel=error", cwd=APP_DIR / "frontend")
run("npm run build", cwd=APP_DIR / "frontend", env={**os.environ, "VITE_API_URL": ""})

print("4/5 Descargando el modelo de IA (YOLOv8)…")
WEIGHTS = BASE / "weights" / "yolov8n.pt"
WEIGHTS.parent.mkdir(exist_ok=True)
if not WEIGHTS.exists():
    urllib.request.urlretrieve(
        "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt", WEIGHTS
    )

print("5/5 Descargando el túnel para el enlace público…")
CLOUDFLARED = BASE / "cloudflared"
if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED,
    )
    CLOUDFLARED.chmod(0o755)

gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("\n✅ Instalado." + (f" GPU: {gpu}" if gpu else
      " ⚠️ Sin GPU: funcionará, pero más lento (cambia el entorno a GPU T4)."))

In [ ]:
#@title 3. Arrancar la app y obtener el enlace { display-mode: "form" }
import re, time, urllib.request
from IPython.display import HTML, display

LOCAL_URL = "http://127.0.0.1:7860"

# Si ya estaba arrancada (re-ejecutando la celda), se para primero.
for proc_name in ("server", "tunnel"):
    proc = globals().get(proc_name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        proc.wait(timeout=10)

env = {
    **os.environ,
    "BALONMANO_DATA_DIR": str(DATA_DIR),
    "BALONMANO_FRONTEND_DIR": str(APP_DIR / "frontend" / "dist"),
    "BALONMANO_YOLO_MODEL_PATH": str(WEIGHTS),
    "BALONMANO_ACCESS_PASSWORD": CONTRASENA,
    "YOLO_CONFIG_DIR": str(BASE / "ultralytics"),
}
LOG = open(BASE / "app.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "7860"],
    cwd=APP_DIR / "backend", env=env, stdout=LOG, stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        urllib.request.urlopen(LOCAL_URL + "/health", timeout=2)
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("La app no arrancó:\n" + (BASE / "app.log").read_text()[-3000:])
        time.sleep(1)
else:
    raise RuntimeError("La app tarda demasiado en arrancar:\n" + (BASE / "app.log").read_text()[-3000:])

tunnel = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--no-autoupdate", "--url", LOCAL_URL],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
PUBLIC_URL = None
deadline = time.time() + 60
while time.time() < deadline and PUBLIC_URL is None:
    line = tunnel.stdout.readline()
    if not line and tunnel.poll() is not None:
        break
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        PUBLIC_URL = match.group(0)
if PUBLIC_URL is None:
    raise RuntimeError("No se pudo crear el enlace público; vuelve a ejecutar esta celda.")

# Keep reading the tunnel output in the background so its pipe never fills up.
import threading
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

display(HTML(f"""
<div style="font-size:16px;line-height:1.6">
  ✅ <b>La app está en marcha</b><br>
  Enlace: <a href="{PUBLIC_URL}" target="_blank">{PUBLIC_URL}</a><br>
  Contraseña: <code>{CONTRASENA}</code> (el usuario da igual)<br>
  <small>El enlace puede tardar ~30 s en responder la primera vez. Deja esta pestaña abierta.</small>
</div>"""))

In [ ]:
#@title 4. (Opcional) Analizar vídeos grandes desde Google Drive { display-mode: "form" }
#@markdown Copia los vídeos a la carpeta **BalonmanoIA/importar** de tu Drive
#@markdown (o a `balonmano_data/importar` si no usas Drive) y ejecuta esta celda.
#@markdown Se suben directamente desde aquí, sin el límite de tamaño del enlace público.
MODALIDAD = "piso"  #@param ["piso", "playa"]
import base64, json, mimetypes, uuid

def upload_local(path):
    boundary = uuid.uuid4().hex
    parts = []
    for name, value in (("name", path.stem), ("court_type", MODALIDAD)):
        parts.append(f"--{boundary}\r\nContent-Disposition: form-data; name=\"{name}\"\r\n\r\n{value}\r\n".encode())
    parts.append(
        f"--{boundary}\r\nContent-Disposition: form-data; name=\"file\"; filename=\"{path.name}\"\r\n"
        f"Content-Type: {mimetypes.guess_type(path.name)[0] or 'video/mp4'}\r\n\r\n".encode()
    )
    body = b"".join(parts) + path.read_bytes() + f"\r\n--{boundary}--\r\n".encode()
    request = urllib.request.Request(LOCAL_URL + "/matches/upload", data=body, method="POST")
    request.add_header("Content-Type", f"multipart/form-data; boundary={boundary}")
    request.add_header("Authorization", "Basic " + base64.b64encode(f"colab:{CONTRASENA}".encode()).decode())
    return json.load(urllib.request.urlopen(request))

pending = sorted(p for p in (DATA_DIR / "importar").iterdir()
                 if p.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv"})
if not pending:
    print("No hay vídeos en", DATA_DIR / "importar")
for video in pending:
    match = upload_local(video)
    video.rename(video.with_name(video.name + ".importado"))
    print(f"✅ {video.name} → sesión «{match['name']}» en cola. Míralo en {PUBLIC_URL}/sesiones/{match['id']}")

In [ ]:
#@title 5. (Opcional) Ver el registro de la app { display-mode: "form" }
print((BASE / "app.log").read_text()[-5000:] or "(vacío)")